# 07. 사전학습 언어모델(공개 가중치)로 01 후보 재순위화
train 에 없는 단어(소탐대실·개막장·허난설헌 등, 검증 어절의 약 12.5%)는 지금 모델들이 모른다.
외부 **데이터**는 금지지만 가중치가 공개된 사전학습 모델은 허용 → 대량의 한국어를 이미 배운 언어모델로 01 후보 중 자연스러운 것을 고른다.

1. 문장을 3어절씩 조각으로 나누고, 조각마다 01 이 상위 8개 후보를 냄 (CPU)
2. 언어모델이 `앞 문맥 + 후보` 의 자연스러움 점수 계산 (GPU)
3. 조각마다 `01 점수 + 가중치 × LM 점수` 최대 후보 선택. 가중치 0 = 01 단독

기본 모델: **Qwen/Qwen2.5-1.5B** (Apache-2.0, 한국어 지원, T4 에서 동작). 다른 공개 모델은 `--lm` 으로.
로컬 확인(검증 15문장): 조각 단위 01 = 문장 단위 01 과 거의 같고, 후보 중 최선을 고를 때 상한이 01 보다 약 +0.006.

**GPU 런타임, 첫 셀부터.** 후보 생성은 CPU 작업이라 코어 많은 런타임(L4·A100)이 빠름.

## 0) 준비

In [ ]:
import os
os.chdir('/content')
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
DRIVE = '/content/drive/MyDrive/데이콘/난독화'
!mkdir -p /content/work && rsync -a --exclude '.playwright-mcp' --exclude '__pycache__' --exclude 'open.zip' --exclude 'v2_cache_*.pkl' "$DRIVE/" /content/work/
%cd /content/work
!ls outputs/best_params.json; nproc; nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
def save_back():
    if not os.path.exists(DRIVE):
        drive.mount('/content/drive', force_remount=True)
    !rsync -a --exclude 'v2_cache_*.pkl' /content/work/outputs/ "$DRIVE/outputs/"
    print('saved outputs -> drive')

## 1) 빠른 확인 (검증 100문장)
모델 내려받기(약 3GB) + 동작 확인. 출력의 '상한' 과 가중치별 F1 을 확인.

In [ ]:
!python -u lm_rerank.py --mode val --val_limit 100

## 2) 검증 전체 → `outputs/best_rerank_<모델>.json`
후보·LM 점수는 캐시(`rerank_*.pkl`)로 저장되어, 가중치만 바꿔 다시 돌리면 즉시 끝남.

In [ ]:
!python -u lm_rerank.py --mode val
save_back()

In [ ]:
import pandas as pd
r = pd.read_csv('outputs/rerank_runs.csv')
r[r.n_val == r.n_val.max()].sort_values('f1', ascending=False).head(10)

### (선택) 다른 공개 모델로 비교
예: 더 작은 `Qwen/Qwen2.5-0.5B`, 한국어 전용 `skt/kogpt2-base-v2`(CC-BY-NC-SA, 비상업 개인 연습용).
모델마다 결과 파일 이름이 달라서 덮어쓰지 않음. 후보 캐시는 공유.

In [ ]:
# !python -u lm_rerank.py --mode val --lm Qwen/Qwen2.5-0.5B
# !python -u lm_rerank.py --mode val --lm skt/kogpt2-base-v2
# save_back()

## 3) 01 과 비교

In [ ]:
from collections import Counter
import glob
a = pd.read_csv('outputs/val_pred.csv').set_index('ID'); a['pred'] = a['pred'].fillna('')
print(f"01 noisy-channel      F1 {a.f1.mean():.4f}")
for f in sorted(glob.glob('outputs/val_pred_rerank_*.csv')):
    b = pd.read_csv(f).set_index('ID'); b['pred'] = b['pred'].fillna('')
    ids = a.index.intersection(b.index); d = b.loc[ids, 'f1'] - a.loc[ids, 'f1']
    print(f"{f.split('val_pred_rerank_')[1][:-4]:20s} F1 {b.loc[ids].f1.mean():.4f}  (rows {len(ids)}) | 좋아진 행 {(d>0).sum()} / 나빠진 행 {(d<0).sum()}")
    for i in d.sort_values().index[:2].tolist() + d.sort_values().index[-3:].tolist():
        print(f'   [{d[i]:+.3f}] GT :', b.loc[i, 'output'][:90]); print('           01 :', a.loc[i, 'pred'][:90]); print('           LM :', b.loc[i, 'pred'][:90])

## 4) test 예측 → `outputs/submission_rerank_<모델>.csv`

In [ ]:
!python -u lm_rerank.py --mode test
save_back()

In [ ]:
!python -u lm_rerank.py --mode val --lm Qwen/Qwen2.5-7B
save_back()


### (선택) 다 끝나면 런타임 해제

In [ ]:
from google.colab import runtime; runtime.unassign()